# 02. Разработка пайплайна кандидатогенерации

Цель блокнота — на локальной выборке последовательно построить и
сравнить retrieval-каналы, выбрать способ агрегации и сохранить финальную
модель. 

Финальная схема использует четыре канала: Local E5, Local TF-IDF,
Nearby E5 и Global E5. Их объединённый пул ранжируется Logistic Regression.

## 1. Окружение и данные

In [ ]:
import gc
import os
import sys
from pathlib import Path

# возникали конфликты на MacOs с SSLKEYLOGFILE
os.environ.pop("SSLKEYLOGFILE", None)

import joblib
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

from src.data import (
    SEARCH_COLUMNS,
    build_candidate_items,
    build_validation_queries,
    integer_ids,
    prepare_coordinates,
    split_by_search_context,
)
from src.fusion import (
    FEATURE_NAMES,
    build_quota_fusion,
    build_training_matrix,
    build_weighted_rrf,
    rank_with_model,
    train_linear_blender,
    union_recall,
)
from src.metrics import evaluate_position_matrix
from src.retrieval import (
    load_or_build_global_semantic,
    load_or_build_local_semantic,
    load_or_build_local_tfidf,
    load_or_build_nearby_semantic,
    load_or_build_tfidf,
    load_or_encode_embeddings,
)
from src.text_features import (
    build_lexical_item_text,
    build_query_text,
    build_semantic_item_text,
)

DATA_DIR = PROJECT_ROOT / "dataset"
ARTIFACTS_DIR = PROJECT_ROOT / "artifacts"
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
train = pd.read_parquet(DATA_DIR / "train.parquet")
benchmark_items = pd.read_parquet(DATA_DIR / "benchmark_items.parquet")

train.shape, benchmark_items.shape

## 2. Локальная валидация

Разделяем данные по полному поисковому контексту: тексту, локации,
категории, доставке и фильтрам. Один и тот же контекст целиком попадает
либо в development, либо в validation.

In [ ]:
development_train, validation_rows = split_by_search_context(train)
candidate_items = build_candidate_items(train)
validation_queries = build_validation_queries(validation_rows)

development_query_texts = set(development_train["search_query"])
validation_queries["query_text_seen"] = validation_queries["search_query"].isin(
    development_query_texts
)

pd.Series({
    "development_rows": len(development_train),
    "validation_rows": len(validation_rows),
    "validation_queries": len(validation_queries),
    "candidate_items": len(candidate_items),
    "unseen_query_texts": (~validation_queries["query_text_seen"]).sum(),
})

В сохранённом разбиении получается 70 893 validation-контекста, из них
10 361 имеют новый текст запроса. Это позволяет отдельно следить за
обобщением, а не только за запоминанием популярных формулировок.

## 3. География и текстовые представления

In [ ]:
location_reference_items = pd.concat(
    [
        train[["item_location_id", "item_latitude", "item_longitude"]],
        benchmark_items[["item_location_id", "item_latitude", "item_longitude"]],
    ],
    ignore_index=True,
).drop_duplicates()

(
    validation_queries,
    item_latitudes,
    item_longitudes,
    query_latitudes,
    query_longitudes,
) = prepare_coordinates(
    validation_queries,
    candidate_items,
    location_reference_items,
    development_train,
)

item_ids = candidate_items["item_id"].astype(str).to_numpy(dtype="U16")
item_location_ids = integer_ids(candidate_items["item_location_id"])
item_category_ids = integer_ids(candidate_items["item_category_id"])
query_location_ids = integer_ids(validation_queries["search_location_id"])
query_category_ids = integer_ids(validation_queries["search_category"])

arrays_to_save = {
    "candidate_item_ids.npy": item_ids,
    "candidate_location_ids.npy": item_location_ids,
    "candidate_category_ids.npy": item_category_ids,
    "candidate_latitudes.npy": item_latitudes,
    "candidate_longitudes.npy": item_longitudes,
    "validation_location_ids.npy": query_location_ids,
    "validation_category_ids.npy": query_category_ids,
    "validation_query_latitudes.npy": query_latitudes,
    "validation_query_longitudes.npy": query_longitudes,
}
for filename, values in arrays_to_save.items():
    np.save(ARTIFACTS_DIR / filename, values)

pd.Series({
    "item_coordinate_coverage": np.mean(
        np.isfinite(item_latitudes) & np.isfinite(item_longitudes)
    ),
    "query_coordinate_coverage": np.mean(
        np.isfinite(query_latitudes) & np.isfinite(query_longitudes)
    ),
})

In [ ]:
query_texts = build_query_text(validation_queries)
lexical_item_texts = build_lexical_item_text(candidate_items)
semantic_item_texts = build_semantic_item_text(candidate_items)

pd.Series({
    "queries": len(query_texts),
    "items": len(semantic_item_texts),
    "example_query": query_texts.iloc[0],
})

TF-IDF получает заголовок и параметры объявления. E5 дополнительно получает
первые 500 символов описания. 

## 4. Построение retrieval-каналов

In [ ]:
tfidf_vectorizer, tfidf_item_matrix, tfidf_query_matrix = load_or_build_tfidf(
    lexical_item_texts,
    query_texts,
    ARTIFACTS_DIR / "tfidf_vectorizer.joblib",
    ARTIFACTS_DIR / "tfidf_item_matrix.npz",
    ARTIFACTS_DIR / "tfidf_validation_query_matrix.npz",
)

tfidf_item_matrix.shape, tfidf_query_matrix.shape

In [ ]:
item_embeddings, query_embeddings = load_or_encode_embeddings(
    semantic_item_texts,
    query_texts,
    ARTIFACTS_DIR / "local_item_embeddings_e5_small.npy",
    ARTIFACTS_DIR / "local_validation_query_embeddings_e5_small.npy",
)

item_embeddings.shape, query_embeddings.shape

### 4.1 Global E5 (global = без ограничения по локации)

In [ ]:
global_e5, global_e5_scores = load_or_build_global_semantic(
    item_embeddings,
    query_embeddings,
    ARTIFACTS_DIR / "local_semantic_top100_indices.npy",
    ARTIFACTS_DIR / "local_semantic_top100_scores.npy",
    ARTIFACTS_DIR / "local_semantic_hnsw_m16_e5_small.faiss",
    top_k=100,
)

global_e5.shape

### 4.2 Local E5 и Local TF-IDF (local = в локации с тем же id)

In [ ]:
local_e5, local_e5_scores = load_or_build_local_semantic(
    item_embeddings,
    query_embeddings,
    item_location_ids,
    item_category_ids,
    query_location_ids,
    query_category_ids,
    ARTIFACTS_DIR / "local_semantic_exact_top40_indices.npy",
    ARTIFACTS_DIR / "local_semantic_exact_top40_scores.npy",
    top_k=40,
)

local_tfidf, local_tfidf_scores = load_or_build_local_tfidf(
    tfidf_item_matrix,
    tfidf_query_matrix,
    item_location_ids,
    item_category_ids,
    query_location_ids,
    query_category_ids,
    ARTIFACTS_DIR / "local_tfidf_top40_indices.npy",
    ARTIFACTS_DIR / "local_tfidf_top40_scores.npy",
    top_k=40,
)

local_e5.shape, local_tfidf.shape

### 4.3 Nearby E5 (в радиусе 100 км от запроса)

In [ ]:
nearby_e5 = load_or_build_nearby_semantic(
    item_embeddings,
    query_embeddings,
    item_location_ids,
    item_category_ids,
    query_location_ids,
    query_category_ids,
    item_latitudes,
    item_longitudes,
    query_latitudes,
    query_longitudes,
    ARTIFACTS_DIR / "nearby_semantic_top20_indices.npy",
    top_k=20,
    radius_km=100,
)

pd.Series((nearby_e5 >= 0).sum(axis=1)).describe()

In [ ]:
channels = {
    "local_e5": {"indices": local_e5, "scores": local_e5_scores},
    "local_tfidf": {"indices": local_tfidf, "scores": local_tfidf_scores},
    "nearby_e5": {"indices": nearby_e5},
    "global_e5": {"indices": global_e5, "scores": global_e5_scores},
}

del tfidf_vectorizer, tfidf_item_matrix, tfidf_query_matrix
del item_embeddings, query_embeddings
_ = gc.collect()

## 5. Качество каналов

In [ ]:
relevant_items = validation_queries["relevant_item_ids"].tolist()

channel_results = []
for name, k in [("Global E5", 50), ("Local E5", 40), ("Local TF-IDF", 40), ("Nearby E5", 20)]:
    channel_key = name.lower().replace(" ", "_").replace("tf-idf", "tfidf")
    scores = evaluate_position_matrix(
        channels[channel_key]["indices"], item_ids, relevant_items, k=k
    )
    channel_results.append({
        "method": name,
        "recall": scores.mean(),
        "hit_rate": (scores > 0).mean(),
    })

pd.DataFrame(channel_results)

In [ ]:
pool_scores, pool_sizes = union_recall(channels, item_ids, relevant_items)

pd.Series({
    "union_recall": pool_scores.mean(),
    "union_hit_rate": (pool_scores > 0).mean(),
    "mean_pool_size": pool_sizes.mean(),
    "median_pool_size": np.median(pool_sizes),
})

Объединение четырёх каналов даёт Recall около **0.842** до ограничения в
50 кандидатов. Это потолок текущего набора retrieval-каналов.

## 6. Сравнение способов агрегации

In [ ]:
all_positions = np.arange(len(validation_queries))
tuning_positions, test_positions = train_test_split(
    all_positions,
    test_size=0.5,
    random_state=42,
    stratify=validation_queries["query_text_seen"],
)

quota_indices = build_quota_fusion(
    channels, item_category_ids, query_category_ids, quotas=(30, 5, 10)
)
rrf_indices = build_weighted_rrf(
    channels,
    item_category_ids,
    query_category_ids,
    weights=(3.0, 0.5, 1.5, 0.5),
    rrf_k=30,
)

quota_scores = evaluate_position_matrix(quota_indices, item_ids, relevant_items)
rrf_scores = evaluate_position_matrix(rrf_indices, item_ids, relevant_items)

In [ ]:
X_train, y_train, covered_queries = build_training_matrix(
    tuning_positions,
    relevant_items,
    item_ids,
    channels,
    item_location_ids,
    query_location_ids,
    item_category_ids,
    query_category_ids,
)
linear_blender = train_linear_blender(X_train, y_train)
model_indices = rank_with_model(
    linear_blender,
    len(validation_queries),
    channels,
    item_location_ids,
    query_location_ids,
    item_category_ids,
    query_category_ids,
    query_positions=test_positions,
)
model_scores = evaluate_position_matrix(model_indices, item_ids, relevant_items)

pd.DataFrame({
    "method": ["Quota", "Weighted RRF", "Logistic Regression"],
    "recall_at_50": [
        quota_scores[test_positions].mean(),
        rrf_scores[test_positions].mean(),
        model_scores[test_positions].mean(),
    ],
    "hit_rate_at_50": [
        (quota_scores[test_positions] > 0).mean(),
        (rrf_scores[test_positions] > 0).mean(),
        (model_scores[test_positions] > 0).mean(),
    ],
})

На одинаковой отложенной половине Logistic Regression показывает лучший
Recall@50 — около **0.810**. Бустинг также проверялся отдельно и дал 0.808,
поэтому более простая линейная модель выбрана финальным агрегатором.

In [ ]:
coefficients = pd.DataFrame({
    "feature": FEATURE_NAMES,
    "coefficient": linear_blender.named_steps["logisticregression"].coef_[0],
}).sort_values("coefficient", ascending=False)

coefficients.round(4)

## 7. Обучение финального агрегатора

In [ ]:
del X_train, y_train, linear_blender
_ = gc.collect()

X_final, y_final, covered_queries = build_training_matrix(
    all_positions,
    relevant_items,
    item_ids,
    channels,
    item_location_ids,
    query_location_ids,
    item_category_ids,
    query_category_ids,
)
final_model = train_linear_blender(X_final, y_final)

model_bundle = {"model": final_model, "feature_names": FEATURE_NAMES}
model_path = ARTIFACTS_DIR / "final_linear_blender.joblib"
joblib.dump(model_bundle, model_path)

pd.Series({
    "training_pairs": len(y_final),
    "positive_share": y_final.mean(),
    "covered_queries": covered_queries,
    "model_path": str(model_path),
})

## 8. Итог экспериментов

| Этап | Recall@50 |
|---|---:|
| Глобальный TF-IDF по заголовку | 0.2098 |
| Локация + TF-IDF с параметрами | 0.6187 |
| Local/Global E5 | 0.7332 |
| E5 + TF-IDF + Nearby, фиксированные квоты | 0.7846 |
| Logistic Regression, held-out validation | **0.8096** |

Главный прирост дали точная география и семантический E5-поиск. TF-IDF
дополняет E5 точными формулировками, Nearby E5 возвращает межлокационные
услуги, а линейный агрегатор устойчивее ручных квот и RRF.